# use clio_dev3 kernel

### Example of Converting HTML to CSV (DF), re-transcoding the video with higher compression, and coverting back to html

In [ ]:
#Takes an html and returns DFs (tested by hs on oct 11)
from bs4 import BeautifulSoup
import pandas as pd
from piat_slim import S3Handle
def html2df(url):
    s3 = S3Handle()
    def url_to_s3_bucket_key(url, bucket='clio.corp.adobe.com'):
        return bucket, url.split(bucket)[-1].lstrip('/')
    def format_prompt(prompt):
        return ' - '.join([f"[{k.strip().replace('[','').replace(']','')}] : {v.strip().rstrip(',')}" for c in prompt.split('\n') if ': ' in c for k, v in [c.split(': ', 1)]])
    root = url.split('clio.corp.adobe.com/')[-1].split('/model_image_comparison')[0]
    bucket, key = url_to_s3_bucket_key(url)
    html_path = s3.download_file(bucket=bucket, key=key)
    with open(html_path, 'r') as file:
        soup = BeautifulSoup(file.read(), 'html.parser')
    data = []
    for row in soup.find_all('tr'):
        cells = row.find_all('td')
        if len(cells) == 3:
            prompt = cells[0].get_text(strip=True)
            video1_url = cells[1].find('video')['src'] if cells[1].find('video') else None
            video2_url = cells[2].find('video')['src'] if cells[2].find('video') else None
            production_info = prompt.split('\n')[0]
            full_prompt = f"[{production_info}] - " + format_prompt(prompt)
            if video1_url and not video1_url.startswith('http'):
                video1_url = f'https://s3.us-west-2.amazonaws.com/clio.corp.adobe.com/{root}/{video1_url.lstrip("./")}'
            if video2_url and not video2_url.startswith('http'):
                video2_url = f'https://s3.us-west-2.amazonaws.com/clio.corp.adobe.com/{root}/{video2_url.lstrip("./")}'
            data.append({'prompt': full_prompt, 'video1': video1_url, 'video2': video2_url})
    return pd.DataFrame(data)
url = 'http://clio.corp.adobe.com/files/merged_content_2024-10-10-21-19-01-4106f80832_md5_c1daee86656c227b8fd63c608e732426.html'
df = html2df(url)
print(df.iloc[0].video1)
print(df.iloc[0].prompt)
df.head()

In [ ]:
# same as above. This takes html and returns DF. But it works with URLs below
from bs4 import BeautifulSoup
import pandas as pd
from piat_slim import S3Handle

def html2df_hs_v1(url):
    s3 = S3Handle()

    def url_to_s3_bucket_key(url, bucket='clio.corp.adobe.com'):
        return bucket, url.split(bucket)[-1].lstrip('/')

    def format_prompt(prompt):
        return ' - '.join([f"[{k.strip().replace('[','').replace(']','')}] : {v.strip().rstrip(',')}" for c in prompt.split('\n') if ': ' in c for k, v in [c.split(': ', 1)]])

    root = url.split('clio.corp.adobe.com/')[-1].split('/model_image_comparison')[0]
    
    if 'kalarot-clio-data' in url:
        bucket = 'kalarot-clio-data'
        key = url.split(bucket)[-1].lstrip('/')
    else:
        bucket, key = url_to_s3_bucket_key(url)

    html_path = s3.download_file(bucket=bucket, key=key)
    
    with open(html_path, 'r') as file:
        soup = BeautifulSoup(file.read(), 'html.parser')

    data = []
    rows = soup.find_all('tr')

    # Debug: check if rows are found
    print(f"Found {len(rows)} rows in the HTML.")
    
    for row in rows:
        cells = row.find_all('td')
        
        # Debug: check if the row contains the expected number of cells
        print(f"Found {len(cells)} cells in this row.")
        
        # Assuming cell[0] contains the prompt and cell[1] contains the video URLs or other content
        prompt = cells[0].get_text(strip=True)
        video_urls = cells[1].find_all('video')

        video1_url = None
        video2_url = None

        # Check if the videos exist and have a 'src' attribute
        if len(video_urls) > 0 and 'src' in video_urls[0].attrs:
            video1_url = video_urls[0]['src']
        if len(video_urls) > 1 and 'src' in video_urls[1].attrs:
            video2_url = video_urls[1]['src']

        # Ensure URLs are complete and correct
        if video1_url and not video1_url.startswith('http'):
            video1_url = f'https://s3.us-west-2.amazonaws.com/{bucket}/{root}/{video1_url.lstrip("./")}'
        if video2_url and not video2_url.startswith('http'):
            video2_url = f'https://s3.us-west-2.amazonaws.com/{bucket}/{root}/{video2_url.lstrip("./")}'

        # Prepare full prompt and store data
        production_info = prompt.split('\n')[0]
        full_prompt = f"[{production_info}] - " + format_prompt(prompt)

        # Debug: check the final prompt and video URLs before adding them to data
        print(f"Full Prompt: {full_prompt}, Video1: {video1_url}, Video2: {video2_url}")

        data.append({'prompt': full_prompt, 'video1': video1_url, 'video2': video2_url})

    # Return DataFrame and also debug the number of entries
    print(f"Data contains {len(data)} rows.")
    return pd.DataFrame(data)

# Test with different URLs
# url = 'https://s3.us-west-2.amazonaws.com/kalarot-clio-data/files/reencoded_videos_md5_ac871b9ccb29ce914e7a1b4c35f6c610.html'
# df = html2df(url)
# df.head()


In [ ]:
# function to re-encode the video. src can be both url and file path and it takes a filter ex: yadif,deshake,deflicker,minterpolate 
import os
import subprocess
from piat_slim import S3Handle
import uuid

def reencode_video(src_video, output_bucket='kalarot-clio-data', input_bucket=None, compression_level=32, fps=24, filter=None, temp_folder='/tmp/'):
    s3 = S3Handle()
    
    # Download the video if it's not a local file
    if not os.path.isfile(src_video):
        known_buckets = ['clio.corp.adobe.com', 'kalarot-clio-data']
        bucket = input_bucket or next((b for b in known_buckets if b in src_video), None)
        key = src_video.split(bucket)[-1].lstrip('/')
        src_video = s3.download_file(bucket=bucket, key=key, destination_folder=temp_folder)
    
    assert os.path.isfile(src_video)
    
    # Generate a local output video file path
    local_output_video = f'{temp_folder}/{uuid.uuid4()}.mp4'
    
    # Construct the ffmpeg command
    if filter:  # If a filter is specified, add it to the ffmpeg command
        command = f'ffmpeg -y -i {src_video} -vf {filter} -c:v libx264 -crf {compression_level} -r {fps} -movflags faststart -loglevel quiet {local_output_video}'
    else:  # If no filter is specified, use the base command
        command = f'ffmpeg -y -i {src_video} -c:v libx264 -crf {compression_level} -r {fps} -movflags faststart -loglevel quiet {local_output_video}' 
    
    # Run the ffmpeg command and check for success
    if subprocess.run(command.split()).returncode != 0:
        return False
    
    # Upload the re-encoded video to S3
    uploaded_key = s3.upload_file(file_path=local_output_video, folder='reencoded_videos', bucket=output_bucket, add_md5=True)
    
    # Return the S3 URL of the uploaded file
    return f'https://s3.us-west-2.amazonaws.com/{output_bucket}/{uploaded_key}'

# Example usage
# reencode_video(src_video='/sensei-fs/users/hshariati/del/clineto/flickering_videos/static1.mp4', filter='deflicker', output_bucket='kalarot-clio-data')
# or url:
# src_video_url= 'http://clio.corp.adobe.com/model_evaluation//sensei-fs/users/hshariati/del/clineto/html/baseline_evalEndPoint_11-08-24_2/286e1491-8a57-4c73-be74-e6b6e0b0f70d.mp4'


In [ ]:
# function to re-encode a df (csv)
import pandas as pd
from tqdm import tqdm
from piat_slim import S3Handle

def reencode_df(df, video_columns=[],compression_level=32):
    df = df.copy()
    assert set(video_columns).issubset(df.columns)
    for index, row in tqdm(df.iterrows(), total=len(df)):
        for video_column in video_columns:
            df.at[index, video_column] = reencode_video(row[video_column],compression_level=compression_level)
    return df
video_columns = ['video2'] #Edit your df's video_columns here 
dfr = reencode_df(df.sample(2), video_columns)
dfr.head()

In [ ]:
# convert DF to html
def df_to_html(df: pd.DataFrame, img_columns: list = [], video_columns: list = [], width: int = 512, file_name: str = 'media.html') -> str:
    df = df.copy()
    for col in img_columns: df[col] = df[col].apply(lambda url: f'<img src="{url}" width="{width}">')
    for col in video_columns: df[col] = df[col].apply(lambda url: f'<video width="{width}" controls autoplay muted><source src="{url}" type="video/mp4"></video>')
    html = df.to_html(escape=False).replace('border="1"', 'border="0"')
    with open(file_name, 'w') as f: f.write(html)
    s3 = S3Handle()
    return f'https://s3.us-west-2.amazonaws.com/kalarot-clio-data/{s3.upload_file(file_name)}'
# df_to_html(dfr,video_columns=video_columns)

In [ ]:
# this takes a folder of videos as input, re-encodes them, stores the results in a DF, and then renders the DF into HTML
# filters can be: yadif,deshake,deflicker,minterpolate 
import os
import pandas as pd
from tqdm import tqdm

# Function to scan a folder for video files
def get_video_files_from_folder(folder_path, extensions=['.mp4', '.mkv', '.avi']):
    video_files = []
    for root, _, files in os.walk(folder_path):
        for file in files:
            if any(file.endswith(ext) for ext in extensions):
                video_files.append(os.path.join(root, file))
    return video_files

# Main function to re-encode videos from a folder and generate HTML
def process_videos_from_folder(folder_path, output_bucket='kalarot-clio-data', compression_level=32, fps=24, filter=None, width=512, html_file_name='reencoded_videos.html'):
    video_files = get_video_files_from_folder(folder_path)
    data = []

    # Re-encode each video
    for video_file in tqdm(video_files):
        reencoded_url = reencode_video(video_file, output_bucket=output_bucket, compression_level=compression_level,fps=fps,filter=filter)
        data.append({'original_video': video_file, 'reencoded_video': reencoded_url})

    # Create a DataFrame
    df = pd.DataFrame(data)

    # Render DataFrame as HTML with embedded videos
    html_url = df_to_html(df, video_columns=['reencoded_video'], width=width, file_name=html_file_name)

    return html_url

# Example usage
folder_path = '/sensei-fs/users/hshariati/del/clineto/flickering_videos'  # Change this to the folder containing your videos
html_url = process_videos_from_folder(folder_path, filter=None, compression_level=40)
print(f'HTML output: {html_url}')


In [ ]:
# another df to html
import pandas as pd
from piat_slim import S3Handle

def df_to_html(df: pd.DataFrame, img_columns: list = [], video_columns: list = [], width: int = 512, file_name: str = 'media.html') -> str:
    df = df.copy()
    
    # Ensure unique column names
    df.columns = [f'{col}_{i}' if df.columns.tolist().count(col) > 1 else col for i, col in enumerate(df.columns)]
    
    # Apply HTML img and video embedding
    for col in img_columns:
        if col in df.columns:
            df[col] = df[col].apply(lambda url: f'<img src="{url}" width="{width}">')
    for col in video_columns:
        if col in df.columns:
            df[col] = df[col].apply(lambda url: f'<video width="{width}" controls autoplay muted><source src="{url}" type="video/mp4"></video>')
    
    # Convert to HTML and save
    html = df.to_html(escape=False).replace('border="1"', 'border="0"')
    with open(file_name, 'w') as f:
        f.write(html)
    
    # Upload to S3
    s3 = S3Handle()
    return f'https://s3.us-west-2.amazonaws.com/kalarot-clio-data/{s3.upload_file(file_name)}'

# Example usage:
video_columns = ['Original', 'CRF-32', 'Deflicker-32']
url = df_to_html(final_df2, video_columns=video_columns)

In [ ]:
# load csv into df
import pandas as pd

# Specify the path to your CSV file
csv_file_path = 'final_hs_dfs_c2.csv'

# Load the CSV file into a DataFrame
final_df2 = pd.read_csv(csv_file_path)

# Display the first few rows of the DataFrame
print(final_df2.head())

## combine htmls column wise

In [ ]:
baseline_url = 'http://clio.corp.adobe.com/model_evaluation//sensei-fs/users/siddhantg/dumps/ref_decoder/flickering/t2v/output_videos_vae3d_decoder_baseline/t2v.html'
ref_dec_url = 'https://clio.corp.adobe.com/model_evaluation//sensei-fs/users/siddhantg/dumps/ref_decoder/flickering/t2v/output_videos/t2v.html'

In [ ]:
# Make sure to delete the v2v.html between calls to html2df otherwise things get overriden.

In [ ]:
# Assuming df1 and df2 are your two DataFrames
# Selecting first two columns from df1 and second column from df2
combined_df = pd.concat([df_baseline.iloc[:, :2], df_ref_dec.iloc[:, 1]], axis=1)

# combined_df will now have the first two columns from df1 and the second column from df2

# this will set the columns to these names
combined_df.columns = ['New_Column1', 'New_Column2', 'New_Column3']

In [ ]:
combined_df.head()

In [ ]:
# Assuming 'combined_df' is your DataFrame
combined_df.columns = ['prompt', 'production', 'ref_decoder_V2']


In [ ]:
# another df to html
import pandas as pd
from piat_slim import S3Handle

def df_to_html(df: pd.DataFrame, img_columns: list = [], video_columns: list = [], width: int = 512, file_name: str = 'media.html') -> str:
    df = df.copy()
    
    # Ensure unique column names
    df.columns = [f'{col}_{i}' if df.columns.tolist().count(col) > 1 else col for i, col in enumerate(df.columns)]
    
    # Apply HTML img and video embedding
    for col in img_columns:
        if col in df.columns:
            df[col] = df[col].apply(lambda url: f'<img src="{url}" width="{width}">')
    for col in video_columns:
        if col in df.columns:
            df[col] = df[col].apply(lambda url: f'<video width="{width}" controls autoplay muted><source src="{url}" type="video/mp4"></video>')
    
    # Convert to HTML and save
    html = df.to_html(escape=False).replace('border="1"', 'border="0"')
    with open(file_name, 'w') as f:
        f.write(html)
    
    # Upload to S3
    s3 = S3Handle()
    return f'https://s3.us-west-2.amazonaws.com/kalarot-clio-data/{s3.upload_file(file_name)}'

# Example usage:
video_columns = ['production', 'ref_decoder_V2']
url = df_to_html(df_comb_2, video_columns=video_columns)

In [ ]:
url

In [ ]:
# Assuming 'df' is your DataFrame
new_comb = combined_df.drop(combined_df.columns[0], axis=1)


In [ ]:
c.head()

In [ ]:
new_comb.iloc[0,0]

In [ ]:
# Assuming 'df' is your DataFrame with two columns of URLs
df_comb_2 = new_comb.apply(lambda col: col.str.replace('/v2v.html', '', regex=False))


In [ ]:
df_comb_2.iloc[0,0]

## other examples

In [ ]:
# this takes a folder of videos as input, re-encodes them, stores the results in a DF, and then renders the DF into HTML
# filters can be: yadif,deshake,deflicker,minterpolate 
import os
import pandas as pd
from tqdm import tqdm

# Function to scan a folder for video files
def get_video_files_from_folder(folder_path, extensions=['.mp4', '.mkv', '.avi']):
    video_files = []
    for root, _, files in os.walk(folder_path):
        for file in files:
            if any(file.endswith(ext) for ext in extensions):
                video_files.append(os.path.join(root, file))
    return video_files

# Main function to re-encode videos from a folder and generate HTML
def process_videos_from_folder(folder_path, output_bucket='kalarot-clio-data', compression_level=32, fps=24, filter=None, width=512, html_file_name='reencoded_videos.html'):
    video_files = get_video_files_from_folder(folder_path)
    data = []

    # Re-encode each video
    for video_file in tqdm(video_files):
        reencoded_url = reencode_video(video_file, output_bucket=output_bucket, compression_level=compression_level,fps=fps,filter=filter)
        data.append({'original_video': video_file, 'reencoded_video': reencoded_url})

    # Create a DataFrame
    df = pd.DataFrame(data)
    final_hs_dfs.append(df)

# Example usage
folder_path = '/sensei-fs/users/hshariati/del/clineto/flickering_videos'  # Change this to the folder containing your videos
process_videos_from_folder(folder_path, filter='minterpolate')

In [ ]:
# get video info
import json
import os
import subprocess
from piat_slim import S3Handle
def get_video_info(src_video, temp_folder='/tmp/', input_bucket=None):
    if not os.path.isfile(src_video):
        s3 = S3Handle()
        known_buckets = ['clio.corp.adobe.com', 'kalarot-clio-data']
        bucket = input_bucket or next((b for b in known_buckets if b in src_video), None)
        key = src_video.split(bucket)[-1].lstrip('/')
        src_video = s3.download_file(bucket=bucket, key=key, destination_folder=temp_folder)
    assert os.path.isfile(src_video)
    probe = f"ffprobe -v quiet -print_format json -show_format -show_streams -i {src_video}"
    result = subprocess.run(probe.split(), stdout=subprocess.PIPE, stderr=subprocess.PIPE)
    return json.loads(result.stdout.decode())

src_video_url = 'http://clio.corp.adobe.com/model_evaluation//sensei-fs/users/hshariati/del/clineto/html/baseline_evalEndPoint_11-08-24_2/286e1491-8a57-4c73-be74-e6b6e0b0f70d.mp4'
get_video_info(src_video_url)

In [ ]:
from bs4 import BeautifulSoup
import pandas as pd
import sys
sys.path.append("/sensei-fs/users/kalarot/git/piat_slim")
def html2df(url):
    from s3_handle import S3Handle
    s3 = S3Handle()
    def url_to_s3_bucket_key(url,bucket = 'clio.corp.adobe.com'):
        key = url.split(bucket)[-1].lstrip('/')
        return bucket, key
    def split_prompt_components_v2(prompt):
        components = prompt.split('\n')
        result = {}
        for component in components:
            if ': ' in component:
                key, value = component.split(': ', 1)
                key = key.replace('[', '').replace(']', '')  # Removing square brackets
                if 'prompt' in key:
                    key = 'prompt'
                result[key.strip()] = value.strip().rstrip(',')
        return result
    root = url.split('clio.corp.adobe.com/')[-1].split('/model_image_comparison')[0]
    bucket, key= url_to_s3_bucket_key(url)
    html_path= s3.download_file('temp/'+root,bucket=bucket,key=key)
    with open(html_path, 'r') as file:
        soup = BeautifulSoup(file.read(), 'html.parser')
    data = []
    for i, row in enumerate(soup.find_all('tr')):
        cells = row.find_all('td')
        if len(cells) > 1:
            prompt = cells[0].get_text(strip=True)
            image_url = cells[1].find('img')['src'] if cells[1].find('img') else None
            prompt_data = split_prompt_components_v2(prompt)
            prompt_data['image'] = image_url
            data.append(prompt_data)

    df = pd.DataFrame(data)
    df
    df['image'] = df['image'].str.replace('./', '/', regex=False)
    base ='https://s3.us-west-2.amazonaws.com/clio.corp.adobe.com/'
    df['image'] =base+root+'/'+df['image']
    return df
dfr= html2df('http://clio.corp.adobe.com/model_comparison/image_comparison_output/2023-12-03-22-54-39-c8e940783c/model_image_comparison.html')
dfr.head()

In [ ]:
# given S3 URL of video, download it to disk. Use: clio_dev3 kernel
import hashlib
import os
import random
import re
import string
from datetime import datetime
from pathlib import Path
from urllib.parse import urlparse
import re
import subprocess
from enum import Enum
import cv2
import numpy as np

import requests
from bs4 import BeautifulSoup
from s3_handle import S3Handle
s3 = S3Handle()

def html2df(url):
    name = url.split('/')[-1][:-5]
    root = url.split('clio.corp.adobe.com/')[-1].split(name)[0]
    bucket, key= url_to_s3_bucket_key(url)
    html_path= s3.download_file('temp/'+root,bucket=bucket,key=key)
    with open(html_path, 'r') as file:
        soup = BeautifulSoup(file.read(), 'html.parser')
    data = []
    for i, row in enumerate(soup.find_all('tr')):
        cells = row.find_all('td')
        if len(cells) > 1:
            prompt = cells[0].get_text(strip=True)
            image_url = cells[1].find('video')['src'] if cells[1].find('video') else None
            prompt_data = split_prompt_components_v2(prompt)
            prompt_data['video'] = image_url
            data.append(prompt_data)

    df = pd.DataFrame(data)
    df['video'] = df['video'].str.replace('./', '/', regex=False)
    base ='https://s3.us-west-2.amazonaws.com/clio.corp.adobe.com/'
    df['video'] =base+root+'/'+df['video']
    return df

def download_video_from_s3(key, temp_dir='./tmp'):
    if 's3.us-west-2.amazonaws.com' in key or 's3-accelerate.amazonaws.com' in key:
        os.makedirs(temp_dir, exist_ok=True)
        local_filename = os.path.join(temp_dir, os.path.basename(key.split('?')[0]))
        response = requests.get(key, stream=True)
        if response.status_code == 200:
            with open(local_filename, 'wb') as f:
                for chunk in response.iter_content(chunk_size=8192):
                    if chunk:
                        f.write(chunk)
            return local_filename
        return None
    else:
        return S3Handle().download_file(bucket='kalarot-clio-data', key=key, destination_folder=temp_dir)

# sandbox

In [45]:
import os
import time
import pandas as pd
import argparse


def wait_for_csv_completion(
    input_folder: str, num_nodes: int, total_inputs: int, sleep_time: int = 10
):
    # Calculate how many lines should be in each CSV file
    lines_per_file = [total_inputs // num_nodes] * num_nodes
    remainder = total_inputs % num_nodes
    for i in range(remainder):
        lines_per_file[i] += 1

    # Paths for job subfolders and CSVs
    job_folders = [os.path.join(input_folder, f"job_{i}") for i in range(num_nodes)]
    csv_paths = [os.path.join(job_folder, "output.csv") for job_folder in job_folders]

    # Wait until all CSVs have the required number of lines
    while True:
        all_files_ready = True
        for i, csv_path in enumerate(csv_paths):
            if not os.path.exists(csv_path):
                all_files_ready = False
                break
            df = pd.read_csv(csv_path)
            if len(df) < lines_per_file[i]:
                all_files_ready = False
                break
        if all_files_ready:
            break
        time.sleep(sleep_time)

    # Concatenate all CSVs once they are complete
    dataframes = [pd.read_csv(csv_path) for csv_path in csv_paths]
    concatenated_df = pd.concat(dataframes, axis=0)

    # Save the concatenated CSV to the input folder
    output_csv_path = os.path.join(input_folder, "final_output.csv")
    concatenated_df.to_csv(output_csv_path, index=False)

    print(f"Final CSV saved at: {output_csv_path}")
    
wait_for_csv_complettion(input_folder='/sensei-fs/users/hshariati/hs_scripts/del/', num_nodes=3, total_inputs=10)